In [0]:
%sql
--  Unity Catalog - catalog, schema, volume creation
CREATE CATALOG IF NOT EXISTS tvmaze;
CREATE SCHEMA IF NOT EXISTS tvmaze.bronze;
CREATE VOLUME IF NOT EXISTS tvmaze.bronze.raw;

In [0]:
# ============================================================
# Imports, Configuration, and Database Setup
# ============================================================

import requests
import json

CONFIG = {
    "api_base_url": "https://api.tvmaze.com",
    "raw_base_path": "/Volumes/tvmaze/bronze/raw",
    "bronze_db": "tvmaze.bronze",
    "tables": {
        "shows": "tvmaze.bronze.tv_shows",
        "episodes": "tvmaze.bronze.episodes",
        "cast": "tvmaze.bronze.cast"
    }
}

spark.sql(f"USE {CONFIG['bronze_db']}")

print("Config loaded")
print(f"Using database: {CONFIG['bronze_db']}")
print(f"Raw path: {CONFIG['raw_base_path']}")


In [0]:
# ============================================================
# API Helper
# ============================================================

def api_get(endpoint: str):
    """Helper function to avoid repeating requests.get()."""
    url = f"{CONFIG['api_base_url']}{endpoint}"
    return requests.get(url).json()

print("API helper ready.")


In [0]:
# ============================================================
# Fetch Shows and Write Raw JSON
# ============================================================

raw_path = f"{CONFIG['raw_base_path']}/shows"
dbutils.fs.mkdirs(raw_path)

shows = api_get("/shows?page=0")

dbutils.fs.put(
    f"{raw_path}/shows.json",
    "\n".join([json.dumps(s) for s in shows]),
    overwrite=True
)

print(f"shows loaded to path: {raw_path}")


In [0]:
# ============================================================
# Fetch Episodes and Write Raw JSON
# ============================================================

raw_path = f"{CONFIG['raw_base_path']}/episodes"
dbutils.fs.mkdirs(raw_path)

episodes = []

for show in shows:
    eps = api_get(f"/shows/{show['id']}/episodes")
    for e in eps:
        e["show_id"] = show["id"]
    episodes.extend(eps)

dbutils.fs.put(
    f"{raw_path}/episodes.json",
    "\n".join([json.dumps(e) for e in episodes]),
    overwrite=True
)

print(f"episodes loaded to path: {raw_path}")


In [0]:
# ============================================================
# Fetch Cast and Write Raw JSON
# ============================================================

raw_path = f"{CONFIG['raw_base_path']}/cast"
dbutils.fs.mkdirs(raw_path)

cast = []

for show in shows:
    c = api_get(f"/shows/{show['id']}/cast")
    for person in c:
        person["show_id"] = show["id"]
    cast.extend(c)

dbutils.fs.put(
    f"{raw_path}/cast.json",
    "\n".join([json.dumps(c) for c in cast]),
    overwrite=True
)

print(f"cast loaded to path: {raw_path}")


In [0]:
# ============================================================
# Write Bronze Delta tables from raw JSON
# ============================================================

def write_bronze_table(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")   # schema enforcement
        .option("mergeSchema", "true")       # schema evolution
        .saveAsTable(table_name)
    )
    print(f"Created Bronze table: {table_name}")


# Shows
shows_df = spark.read.json(f"{CONFIG['raw_base_path']}/shows")
write_bronze_table(shows_df, CONFIG["tables"]["shows"])

# Episodes
episodes_df = spark.read.json(f"{CONFIG['raw_base_path']}/episodes")
write_bronze_table(shows_df, CONFIG["tables"]["episodes"])

# Cast
cast_df = spark.read.json(f"{CONFIG['raw_base_path']}/cast")
write_bronze_table(cast_df, CONFIG["tables"]["cast"])
